In [1]:
import os

os.environ["OPENAI_API_KEY"] = "sk-60a9041004c34c4caffab1616e388989"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [3]:
# 导入所需模块
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1. 创建基础模板
# input_variables 定义了模板中需要填充的变量
template = PromptTemplate(
    input_variables=["product", "target_audience", "tone"],
    template="""请为 {product} 写一个面向 {target_audience} 的产品描述。

要求：
- 语调：{tone}
- 长度：100-200字
- 突出产品优势
- 包含行动号召

产品描述："""
)

# 2. 格式化提示词
# 使用 format 方法填充变量
prompt = template.format(
    product="智能手表",
    target_audience="年轻专业人士",
    tone="专业且友好"
)

print("生成的提示词:")
print(prompt)

# 3. 调用 LLM (可选)
response = llm.invoke(prompt)
print("\\nLLM 响应:")
print(response.content)


生成的提示词:
请为 智能手表 写一个面向 年轻专业人士 的产品描述。

要求：
- 语调：专业且友好
- 长度：100-200字
- 突出产品优势
- 包含行动号召

产品描述：
\nLLM 响应:
专为年轻专业人士设计的智能手表，融合时尚外观与强大功能，助你高效掌控每一天。精准健康监测、消息提醒、运动追踪，一表尽享便捷生活。轻薄设计，适配多种场合，是你职场与生活的得力伙伴。立即选购，开启更智能的工作节奏与生活方式。


In [4]:
# 导入所需模块
from langchain.prompts import ChatPromptTemplate

# 1. 创建聊天模板
# from_messages 接收一个消息列表，每条消息是一个元组 (角色, 内容)
chat_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的 {role}，具有 {experience} 年的经验。你的回答应该准确、专业且易于理解。"),
    ("human", "我有一个关于 {topic} 的问题：{question}"),
    ("ai", "我理解您的问题。让我为您提供专业的解答。"), # 在新版本中，'assistant' 角色通常用 'ai' 表示
    ("human", "{follow_up}")
])

# 2. 格式化消息
# 使用 format_messages 方法填充所有变量
messages = chat_template.format_messages(
    role="数据科学家",
    experience="5",
    topic="机器学习",
    question="什么是过拟合，如何避免？",
    follow_up="能否提供一些具体的防止过拟合的技术？"
)

print("生成的聊天消息:")
for i, message in enumerate(messages):
    print(f"{i+1}. {message.type}: {message.content}")

生成的聊天消息:
1. system: 你是一个专业的 数据科学家，具有 5 年的经验。你的回答应该准确、专业且易于理解。
2. human: 我有一个关于 机器学习 的问题：什么是过拟合，如何避免？
3. ai: 我理解您的问题。让我为您提供专业的解答。
4. human: 能否提供一些具体的防止过拟合的技术？


In [5]:
# 导入所需模块
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

# 1. 准备示例数据
examples = [
    {"input": "这个产品真的很棒！", "output": "正面"},
    {"input": "质量太差了。", "output": "负面"},
    {"input": "还可以吧。", "output": "中性"}
]

# 2. 创建示例的格式化模板
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="输入: {input}\\n输出: {output}"
)

# 3. 创建 Few-shot 模板
few_shot_template = FewShotPromptTemplate(
    examples=examples,              # 示例列表
    example_prompt=example_prompt,  # 格式化示例的模板
    prefix="请根据以下示例，分析文本的情感倾向：", # 提示词前缀
    suffix="输入: {text}\\n输出:", # 提示词后缀，包含最终的输入变量
    input_variables=["text"]        # 最终需要用户输入的变量
)

# 4. 格式化提示词
prompt = few_shot_template.format(text="这次购物体验超出了我的预期！")

print("生成的 Few-shot 提示词:")
print(prompt)

生成的 Few-shot 提示词:
请根据以下示例，分析文本的情感倾向：

输入: 这个产品真的很棒！\n输出: 正面

输入: 质量太差了。\n输出: 负面

输入: 还可以吧。\n输出: 中性

输入: 这次购物体验超出了我的预期！\n输出:


In [6]:
# 导入所需模块
from langchain.prompts import PromptTemplate, PipelinePromptTemplate

# 1. 创建子模板
introduction_template = PromptTemplate.from_template("我是 {name}，一名 {role}。")
task_template = PromptTemplate.from_template("我需要完成以下任务：{task}\\n\\n具体要求：\\n{requirements}")
conclusion_template = PromptTemplate.from_template("请在 {deadline} 之前完成，谢谢！")

# 2. 定义最终模板和管道
final_prompt = PromptTemplate.from_template(
    "{introduction}\\n\\n{task_description}\\n\\n{conclusion}"
)

pipeline_prompts = [
    ("introduction", introduction_template),
    ("task_description", task_template),
    ("conclusion", conclusion_template)
]

# 3. 创建管道模板
pipeline_template = PipelinePromptTemplate(
    final_prompt=final_prompt,
    pipeline_prompts=pipeline_prompts
)

# 4. 格式化提示词
prompt = pipeline_template.format(
    name="张三",
    role="项目经理",
    task="制定下季度的营销计划",
    requirements="1. 分析市场趋势\\n2. 制定推广策略\\n3. 预算规划",
    deadline="下周五"
)

print("生成的管道提示词:")
print(prompt)

生成的管道提示词:
我是 张三，一名 项目经理。\n\n我需要完成以下任务：制定下季度的营销计划\n\n具体要求：\n1. 分析市场趋势\n2. 制定推广策略\n3. 预算规划\n\n请在 下周五 之前完成，谢谢！


/var/folders/pd/wkzj0pvs0zl9s_gx8n1_90h80000gn/T/ipykernel_69625/1396645080.py:21: LangChainDeprecationWarning: This class is deprecated in favor of chaining individual prompts together.
  pipeline_template = PipelinePromptTemplate(


In [7]:
# 导入所需模块
from jinja2 import Environment, FileSystemLoader
from datetime import datetime

# 1. 设置 Jinja2 环境
TEMPLATES_DIR = "template" # 假设模板文件在此目录下
env = Environment(loader=FileSystemLoader(TEMPLATES_DIR))

# 2. 加载模板
template = env.get_template('analysis_report.jinja2')

# 3. 准备数据
data = {
    'report_title': '第三季度销售数据分析',
    'author': '数据分析团队',
    'current_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'include_summary': True,
    'data_points': [
        {'name': '产品A销量', 'value': 150, 'type': '销售', 'trend': 'up'},
        {'name': '产品B销量', 'value': 80, 'type': '销售', 'trend': 'down'},
        {'name': '客户满意度', 'value': 95, 'type': '质量'},
        {'name': '退货率', 'value': 3, 'type': '质量', 'trend': 'stable'}
    ]
}

# 4. 渲染模板
report = template.render(**data)
print(report)

报告标题: 第三季度销售数据分析
作者: 数据分析团队
生成时间: 2025-08-28 11:24:26


## 摘要
以下是关键数据点：

- 产品A销量: 150 (趋势: up)

- 产品B销量: 80 (趋势: down)

- 客户满意度: 95

- 退货率: 3 (趋势: stable)



## 详细分析

- 销售数据点 '产品A销量' 的值为 150。

- 销售数据点 '产品B销量' 的值为 80。


- 质量指标 '客户满意度' 的值为 95。

- 质量指标 '退货率' 的值为 3。

